In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp

import os
import sys
import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec


# Absolute path to cafpyana directory
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *

from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.CutMasks.CutMasks import *
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.GraphUtils.GraphsUtils import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.Optimize import OptimizationUtils
from analysis_village.cc1pi.Optimize import ConfusionMatricesUtils
from analysis_village.cc1pi.BDTs import BDTTrainingUtils

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

np.seterr(divide='ignore', invalid='ignore', over='ignore')

import uproot
import awkward as ak
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, FancyBboxPatch
from matplotlib.colors import to_rgba
from pathlib import Path

In [ ]:
use_range_momentum = True          # base_P = range_P if True else true_P
hits_to_plot = "percent"           # "used", "removed", "percent"
 
hit_bins_used_removed = [0, 50, 100, 200, 300, 500, 750]
hit_bins_percent = [0, 0.25, 0.5, 0.65, 0.8, 0.9, 0.95]
 
filename_vec = [
    "13_MC_subtracted",
    "13_data_subtracted",
]
# index 0 is drawn as "MC" (solid line, circle marker)
# index 1 is drawn as "Data" (dashed line, square marker) -- like `i_f == 1` in C++
file_style_labels = ["MC", "Data"]



input_dir  = Path("/exp/sbnd/data/users/lpelegri/TLE_processed_files")
output_dir = Path("/exp/sbnd/data/users/lpelegri/TLEGraphs/MoneyPlotComp")

In [ ]:

# momentum binning -> LocalBinInformation("momentum", 16, 0, 0.8) in the macro
n_p_bins = 16
p_low, p_high = 0.0, 0.8
p_bin_edges = np.linspace(p_low, p_high, n_p_bins + 1)
p_bin_centers = 0.5 * (p_bin_edges[:-1] + p_bin_edges[1:])
p_bin_halfwidths = 0.5 * np.diff(p_bin_edges)
 
REF_LABEL = "Range" if use_range_momentum else "True"
P_STRING = rf"$P_{{{REF_LABEL}}}$ [GeV]"
 
SENTINEL = -1000.0  # marks an (hit_bin, p_bin) with zero entries, like the C++ code
 
# qualitative, colorblind-friendly palette -- one color per hit bin
_palette = [
    "#0072B2", "#D55E00", "#009E73", "#CC79A7",
    "#E69F00", "#56B4E9", "#999999", "#F0E442",
]
 
hit_bins = hit_bins_percent if hits_to_plot == "percent" else hit_bins_used_removed
n_hit_bins = len(hit_bins)
hit_colors = [_palette[i % len(_palette)] for i in range(n_hit_bins)]

def hit_bin_label(i_hb):
    """Mirrors the legend-string logic of the C++ macro."""
    if hits_to_plot == "percent":
        if i_hb == n_hit_bins - 1:
            return f"Track fraction: > {hit_bins[i_hb]:.2f}"
        return f"Track fraction: {hit_bins[i_hb]:.2f} - {hit_bins[i_hb + 1]:.2f}"
    if i_hb == n_hit_bns - 1:
        return f"{prefix}> {hit_bins[i_hb]}"
    return f"{prefix}{hit_bins[i_hb]} - {hit_bins[i_hb + 1]}"
 
 
def get_hit_bin(values):
    """Vectorised equivalent of GetHitBin(): index of the bin each value falls
    into, right-open, with overflow clamped to the last bin (matching the
    C++ `else return hit_bins.size()-1` overflow behaviour)."""
    edges = np.asarray(hit_bins, dtype=float)
    idx = np.searchsorted(edges[1:], values, side="right")
    return np.minimum(idx, n_hit_bins - 1)

In [ ]:

 
 
# ======================================================================
# 2. READING / FLATTENING THE TREES  (adapted from the uproot example)
# ======================================================================
 
scalar_fields = [
    "true_P", "range_P", "length", "purity", "completeness", "PDG",
    "end_process_string", "out_best_plane", "used_plane",
    "num_used_hits", "num_initial_hits", "num_removed_hits",
]
vector_fields = ["method_vector", "P_track_extension_method", "exit_code"]
 
 
def load_tree_df(file_path):
    """Read one ROOT file and return a flat (event x method) DataFrame with
    all C++-macro-equivalent selections and derived columns already applied."""
    tree = uproot.open(file_path)["tree"]
    arrays = tree.arrays(scalar_fields + vector_fields, library="ak")
    n_events = len(arrays)
    print(f"[{file_path.name}] Entries: {n_events}")
 
    assert ak.all(ak.num(arrays["method_vector"]) == ak.num(arrays["exit_code"]))
    assert ak.all(
        ak.num(arrays["method_vector"]) == ak.num(arrays["P_track_extension_method"])
    )
 
    def to_flat_numpy(ak_array):
        try:
            return ak.to_numpy(ak_array)
        except Exception:
            return np.array(ak.to_list(ak_array), dtype=object)
 
    counts = ak.to_numpy(ak.num(arrays["method_vector"], axis=1))
    data = {"entry": np.repeat(np.arange(n_events), counts)}
    for f in scalar_fields:
        data[f] = np.repeat(to_flat_numpy(arrays[f]), counts)
    for f in vector_fields:
        data[f] = to_flat_numpy(ak.flatten(arrays[f], axis=1))
 
    df = pd.DataFrame(data).rename(columns={
        "PDG": "pdg",
        "method_vector": "method",
        "P_track_extension_method": "reco_P_method",
    })
 
    # -- selection identical to the C++ loop --------------------------------
    df = df[df["exit_code"] == 0].copy()
 
    df["base_P"] = df["range_P"] if use_range_momentum else df["true_P"]

    df["residual"] = (df["reco_P_method"] - df["base_P"]) / df["base_P"]
    df = df[np.isfinite(df["residual"])]
    df = df[(df["residual"] > -1.5) & (df["residual"] < 1.5)].copy()
 
    df["p_type_specific"] = np.where(
        df["end_process_string"].isin(["pipInelastic", "pimInelastic"]),
        "Inelastic",
        "Stopping",
    )
 
    if hits_to_plot == "removed":
        df["hits_metric"] = df["num_removed_hits"]
    elif hits_to_plot == "percent":
        df["hits_metric"] = df["num_used_hits"] / df["num_initial_hits"]
    else:  # "used"
        df["hits_metric"] = df["num_used_hits"]
 
    df["hit_bin"] = get_hit_bin(df["hits_metric"].values)
    df["p_bin"] = np.digitize(df["base_P"].values, p_bin_edges) - 1
    df = df[(df["p_bin"] >= 0) & (df["p_bin"] < n_p_bins)].copy()
 
    return df.reset_index(drop=True)
 
 
dfs = [load_tree_df(input_dir / f"{name}.root") for name in filename_vec]
presets = sorted(set(dfs[0]["method"].unique()) | set(dfs[1]["method"].unique()))
print("Presets found:", presets)
 
 

In [ ]:
from matplotlib.gridspec import GridSpec
'''
def compute_profile(df, preset):
    """Returns (mean, mean_err, std, std_err), each shaped
    [n_hit_bins, n_p_bins] -- the equivalent of filling one TH1D per
    (hit_bin, p_bin) with the residual values and reading off
    GetMean(), GetMeanError(), GetStdDev(), GetStdDevError()."""
 
    sub = df[df["method"] == preset]
 
    mean = np.full((n_hit_bins, n_p_bins), SENTINEL)
    mean_err = np.zeros((n_hit_bins, n_p_bins))
    std = np.full((n_hit_bins, n_p_bins), SENTINEL)
    std_err = np.zeros((n_hit_bins, n_p_bins))
 
    for (i_hb, i_pb), vals in sub.groupby(["hit_bin", "p_bin"])["residual"]:
        n = len(vals)
        if n == 0:
            continue
        s = vals.std(ddof=0)
        mean[i_hb, i_pb] = vals.mean()
        mean_err[i_hb, i_pb] = s / np.sqrt(n)
        std[i_hb, i_pb] = s
        std_err[i_hb, i_pb] = s / np.sqrt(2 * n)
 
    return mean, mean_err, std, std_err
'''

def compute_profile(df, preset, clip_range=None):
    """Returns (mean, mean_err, std, std_err), each shaped
    [n_hit_bins, n_p_bins] -- the equivalent of filling one TH1D per
    (hit_bin, p_bin) with the residual values and reading off
    GetMean(), GetMeanError(), GetStdDev(), GetStdDevError().

    clip_range: if set, only residuals within [-clip_range, clip_range]
    are included in the mean/std for a given (hit_bin, p_bin) group.
    If None, all residuals in the group are used (no clipping).
    """

    sub = df[df["method"] == preset]

    mean = np.full((n_hit_bins, n_p_bins), SENTINEL)
    mean_err = np.zeros((n_hit_bins, n_p_bins))
    std = np.full((n_hit_bins, n_p_bins), SENTINEL)
    std_err = np.zeros((n_hit_bins, n_p_bins))

    for (i_hb, i_pb), vals in sub.groupby(["hit_bin", "p_bin"])["residual"]:
        if clip_range is not None:
            vals = vals[(vals >= -clip_range) & (vals <= clip_range)]
        n = len(vals)
        if n == 0:
            continue
        s = vals.std(ddof=0)
        mean[i_hb, i_pb] = vals.mean()
        mean_err[i_hb, i_pb] = s / np.sqrt(n)
        std[i_hb, i_pb] = s
        std_err[i_hb, i_pb] = s / np.sqrt(2 * n)

    return mean, mean_err, std, std_err
    
def build_legend_handles(overlay=True):
    handles = [
        Line2D([0], [0], color=hit_colors[i], lw=2, label=hit_bin_label(i))
        for i in range(n_hit_bins)
    ]
    if overlay:
        handles.append(Line2D([0], [0], color="black", marker="o", linestyle="-",
                               label=file_style_labels[0]))
        handles.append(Line2D([0], [0], color="black", marker="s", linestyle="--",
                               label=file_style_labels[1]))
    return handles
 

def plot_profile_overlay(values, errs, ax, y_range, y_label, title, show_xlabel=True):
    for i_hb in range(n_hit_bins):
        for i_f in range(2):
            v, e = values[i_f][i_hb], errs[i_f][i_hb]
            mask = v != SENTINEL
            marker = "o" if i_f == 0 else "s"
            linestyle = "-" if i_f == 0 else "--"
            ax.errorbar(
                p_bin_centers[mask], v[mask], yerr=e[mask],
                xerr=p_bin_halfwidths[mask], marker=marker, linestyle=linestyle,
                color=hit_colors[i_hb], markersize=5, capsize=2, linewidth=1.2,
            )
    ax.axhline(0, color="gray", linestyle="--", linewidth=1.5, zorder=0)
    if show_xlabel:
        ax.set_xlabel(P_STRING, fontsize=13)
    ax.set_ylabel(y_label, fontsize=13)
    ax.set_ylim(*y_range)
    ax.set_title(title, fontsize=13)
    ax.grid(alpha=0.25)


def plot_difference_overlay(values, errs, ax, y_label, title=None, show_xlabel=True):
    for i_hb in range(n_hit_bins):
        v0, v1 = values[0][i_hb], values[1][i_hb]
        e0, e1 = errs[0][i_hb], errs[1][i_hb]
        mask = (v0 != SENTINEL) & (v1 != SENTINEL)
        diff = v0[mask] - v1[mask]
        err = np.sqrt(e0[mask] ** 2 + e1[mask] ** 2)
        ax.errorbar(
            p_bin_centers[mask], diff, yerr=err, xerr=p_bin_halfwidths[mask],
            marker="o", linestyle="none", color=hit_colors[i_hb],
            markersize=5, capsize=2,
        )
    ax.axhline(0, color="gray", linestyle="--", linewidth=1.5, zorder=0)
    if show_xlabel:
        ax.set_xlabel(P_STRING, fontsize=13)
    ax.set_ylabel(y_label, fontsize=13)
    ax.set_ylim(-0.2, 0.2)
    if title:
        ax.set_title(title, fontsize=13)
    ax.grid(alpha=0.25)


def plot_combined(
    values, errs, y_range, overlay_label, diff_label, title, legend_handles, metrics=None
):
    """One figure: overlay panel on top (70% height), difference panel below (30%),

    sharing the x-axis.
    """
    fig = plt.figure(figsize=(7, 8.5))
    gs = GridSpec(2, 1, height_ratios=[7, 3], hspace=0.08, figure=fig)

    ax_top = fig.add_subplot(gs[0])
    ax_bot = fig.add_subplot(gs[1], sharex=ax_top)

    plot_profile_overlay(
        values, errs, ax_top, y_range, overlay_label, title, show_xlabel=False
    )
    plt.setp(ax_top.get_xticklabels(), visible=False)

    plot_difference_overlay(
        values, errs, ax_bot, diff_label, title=None, show_xlabel=True
    )

    ax_top.legend(handles=legend_handles, fontsize=8, loc="lower left")

    # Add agreement metric box to the difference panel
    if metrics is not None:
        text_str = (
            rf"$\chi^2/\mathrm{{ndof}} = {metrics['chi2_red']:.2f}\ ({metrics['chi2']:.1f}/{metrics['ndof']})$"
            "\n"
            rf"$p\text{{-val}} = {metrics['p_val']:.3f}$"
            "\n"
            f"RMSD = {metrics['rmsd']:.4f}"
        )
        ax_bot.text(
            0.97,
            0.92,
            text_str,
            transform=ax_bot.transAxes,
            fontsize=8,
            verticalalignment="top",
            horizontalalignment="right",
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.8, edgecolor="gray"),
        )

    return fig


stat_defs = [
    # folder, stat key, overlay y-range, overlay label, diff label
    (
        "Bias", "mean", (-1, 0.2),
        rf"$\mu\left(\frac{{P_{{Reco}}-P_{{{REF_LABEL}}}}}{{P_{{{REF_LABEL}}}}}\right)$",
        r"$\mu_{MC} - \mu_{Reco}$",
    ),
    (
        "StdDev", "std", (0, 0.5),
        rf"$\sigma\left(\frac{{P_{{Reco}}-P_{{{REF_LABEL}}}}}{{P_{{{REF_LABEL}}}}}\right)$",
        r"$\sigma_{MC} - \sigma_{Reco}$",
    ),
]

import numpy as np
from scipy.stats import chi2


def compute_agreement_metrics(v0, e0, v1, e1, sentinel=SENTINEL):
    """Computes Chi2/ndof, p-value, and RMSD across all valid (hit_bin, p_bin) cells."""
    # Mask out SENTINEL values and zero-uncertainty bins
    mask = (v0 != sentinel) & (v1 != sentinel) & ((e0**2 + e1**2) > 0)

    if not np.any(mask):
        return None

    diff = v0[mask] - v1[mask]
    var_sum = e0[mask] ** 2 + e1[mask] ** 2

    chi2_val = np.sum((diff**2) / var_sum)
    ndof = np.sum(mask)
    chi2_red = chi2_val / ndof if ndof > 0 else np.nan
    p_val = chi2.sf(chi2_val, ndof) if ndof > 0 else np.nan
    rmsd = np.sqrt(np.mean(diff**2))

    return {
        "chi2": chi2_val,
        "ndof": ndof,
        "chi2_red": chi2_red,
        "p_val": p_val,
        "rmsd": rmsd,
    }
    

legend_overlay = build_legend_handles(overlay=True)

for preset in presets:
    profiles = [compute_profile(dfs[i_f], preset, clip_range=1.5) for i_f in range(2)]
    mean_vals = [profiles[i_f][0] for i_f in range(2)]
    mean_errs = [profiles[i_f][1] for i_f in range(2)]
    std_vals = [profiles[i_f][2] for i_f in range(2)]
    std_errs = [profiles[i_f][3] for i_f in range(2)]

    for folder, stat, y_range, overlay_label, diff_label in stat_defs:
        values = mean_vals if stat == "mean" else std_vals
        errs = mean_errs if stat == "mean" else std_errs

        # Compute agreement metrics between df[0] and df[1]
        metrics = compute_agreement_metrics(values[0], errs[0], values[1], errs[1])

        out_dir = output_dir / folder
        out_dir.mkdir(parents=True, exist_ok=True)

        fig = plot_combined(
            values,
            errs,
            y_range,
            overlay_label,
            diff_label,
            preset,
            legend_overlay,
            metrics=metrics,
        )
        fig.savefig(out_dir / f"{preset}.pdf")


        if metrics:
            print(
                f"[{preset} - {stat}] Chi2/ndof: {metrics['chi2_red']:.2f}, "
                f"p-val: {metrics['p_val']:.4f}, RMSD: {metrics['rmsd']:.4f}"
            )

print(f"Done. Plots written under: {output_dir}")

In [ ]:
DATA_IDX = 1  # index into dfs / file_style_labels for the data file
MC_IDX = 0    # index into dfs / file_style_labels for the MC file

# Auto-detect the shifted / og convolution presets from the presets list
shifted_preset = next(p for p in presets if "convolution" in p and "shifted" in p)
og_preset = next(p for p in presets if "convolution" in p and "og" in p)
print(f"Data preset: {shifted_preset}   MC preset: {og_preset}")

data_shifted_profile = compute_profile(dfs[DATA_IDX], shifted_preset, clip_range=1.5)
mc_og_profile = compute_profile(dfs[MC_IDX], og_preset, clip_range=1.5)

# Reassemble into the [file][hit_bin] shape the existing plot functions expect,
# with index 0 = data/shifted, index 1 = mc/og.
mean_vals_cmp = [data_shifted_profile[0], mc_og_profile[0]]
mean_errs_cmp = [data_shifted_profile[1], mc_og_profile[1]]
std_vals_cmp = [data_shifted_profile[2], mc_og_profile[2]]
std_errs_cmp = [data_shifted_profile[3], mc_og_profile[3]]

# Legend should read "Data (shifted)" / "MC (og)" rather than generic labels
legend_cmp = [
    Line2D([0], [0], color=hit_colors[i], lw=2, label=hit_bin_label(i))
    for i in range(n_hit_bins)
] + [
    Line2D([0], [0], color="black", marker="o", linestyle="-", label="Data (shifted)"),
    Line2D([0], [0], color="black", marker="s", linestyle="--", label="MC (og)"),
]

for folder, stat, y_range, overlay_label, diff_label in stat_defs:
    values = mean_vals_cmp if stat == "mean" else std_vals_cmp
    errs = mean_errs_cmp if stat == "mean" else std_errs_cmp

    # Compute agreement metrics between Data (shifted) [0] and MC (og) [1]
    metrics = compute_agreement_metrics(values[0], errs[0], values[1], errs[1])

    out_dir = output_dir / folder
    out_dir.mkdir(parents=True, exist_ok=True)

    fig = plot_combined(
        values, errs, y_range, overlay_label, diff_label,
        "Data (shifted) vs MC (og)", legend_cmp,
        metrics=metrics  # Pass metrics to display in the plot
    )
    fig.savefig(out_dir / "data_shifted_vs_mc_og.pdf")

    if metrics:
        print(
            f"[Data(shifted) vs MC(og) - {stat}] Chi2/ndof: {metrics['chi2_red']:.2f}, "
            f"p-val: {metrics['p_val']:.4f}, RMSD: {metrics['rmsd']:.4f}"
        )

print(f"Done. Data-shifted-vs-MC-og comparison written under: {output_dir}")